In [3]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path

# new import
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check

from common.loaders import load_valid_country_ids

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,demonyms"

# --- Esquema de validación ---
demonym_schema = pa.DataFrameSchema({
    "cca3": Column(str, [
        Check.str_length(3, 3),
        Check.isin(load_valid_country_ids())
    ], nullable=False),
    "language": Column(str, [
        Check.isin(["eng", "fra"])  # solo inglés o francés
    ], nullable=False),
    "gender": Column(str, [
        Check.isin(["f", "m"])  # femenino o masculino
    ], nullable=False),
    "demonym": Column(str, nullable=False)
})

@task
def extract_demonyms():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()

@task
def transform_demonyms(data):
    # Aplanar JSON
    df = pd.json_normalize(data, sep=".")
    records = []

    for _, row in df.iterrows():
        cca3 = row["cca3"]

        for lang in ["eng", "fra"]:
            for gender in ["f", "m"]:
                demonym = row.get(f"demonyms.{lang}.{gender}")
                if pd.notna(demonym):
                    records.append({
                        "cca3": cca3,
                        "language": lang,
                        "gender": gender,
                        "demonym": demonym
                    })

    return pd.DataFrame(records)

@task
def validate_demonyms(df: pd.DataFrame) -> pd.DataFrame:
    return demonym_schema.validate(df)

@task
def load_demonyms(df: pd.DataFrame):
    file_path = Path("../stage/country_demonyms.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to {file_path}")

@flow(name="etl_demonyms_flow")
def etl_demonyms():
    data = extract_demonyms()
    df = transform_demonyms(data)
    df = validate_demonyms(df)  # validación con pandera
    load_demonyms(df)

if __name__ == "__main__":
    etl_demonyms()


03:43:49.343 | INFO    | Flow run 'optimal-wallaby' - Beginning flow run 'optimal-wallaby' for flow 'etl_demonyms_flow'

03:43:50.503 | INFO    | Task run 'extract_demonyms-0e4' - Finished in state Completed()

03:43:50.757 | INFO    | Task run 'transform_demonyms-722' - Finished in state Completed()

03:43:51.005 | INFO    | Task run 'validate_demonyms-5ce' - Finished in state Completed()

Saved 1000 rows to ..\stage\country_demonyms.csv


03:43:51.228 | INFO    | Task run 'load_demonyms-ff6' - Finished in state Completed()

03:43:51.407 | INFO    | Flow run 'optimal-wallaby' - Finished in state Completed()